In [ ]:
# =========================================================
# 0) INSTALLS
# =========================================================
!pip install -U "transformers>=4.46.0" "datasets>=3.0.0" "accelerate>=1.0.0" "trl>=1.0.0" "peft>=0.13.0" "bitsandbytes>=0.44.0"

In [66]:
# =========================================================
# 1) IMPORTS
# =========================================================
import gc
import torch
import trl

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)
from trl import RewardTrainer, RewardConfig
from trl.experimental.ppo import PPOTrainer, PPOConfig

print("TRL version:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

TRL version: 1.0.0
CUDA available: True
GPU: Tesla T4


In [67]:
# =========================================================
# 2) HELPERS
# =========================================================
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Optional: helps fragmentation in some environments
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Choose lower precision for memory savings
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [68]:
# =========================================================
# 3) REWARD MODEL TRAINING
# =========================================================
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tok_rm = AutoTokenizer.from_pretrained(model_name)
if tok_rm.pad_token is None:
    tok_rm.pad_token = tok_rm.eos_token

reward_data = [
    {
        "prompt": "How should I prepare for exams?",
        "chosen": "Make a schedule, revise daily, and sleep well.",
        "rejected": "Do nothing and panic at the end.",
    },
    {
        "prompt": "How do I improve fitness?",
        "chosen": "Exercise consistently, eat well, and recover properly.",
        "rejected": "Starve yourself and skip sleep.",
    },
    {
        "prompt": "What is AI?",
        "chosen": "AI is the simulation of intelligent behavior by machines.",
        "rejected": "AI is random magic with no real meaning.",
    },
]

reward_ds = Dataset.from_list(reward_data)

rm = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1,
    torch_dtype=dtype,
)
rm.config.pad_token_id = tok_rm.pad_token_id

rm_args = RewardConfig(
    output_dir="./rm_out",
    per_device_train_batch_size=1,   # lower memory
    per_device_eval_batch_size=1,    # lower memory
    num_train_epochs=1,
    learning_rate=1e-5,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),  # use fp16 if GPU available
)

rm_trainer = RewardTrainer(
    model=rm,
    args=rm_args,
    processing_class=tok_rm,
    train_dataset=reward_ds,
)

print("=== Training Reward Model ===")
rm_trainer.train()

rm_trainer.save_model("./rm_out")
tok_rm.save_pretrained("./rm_out")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Adding EOS to train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Filtering train >1024 tokens:   0%|          | 0/3 [00:00<?, ? examples/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 260.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 17.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 13.84 GiB is allocated by PyTorch, and 583.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# =========================================================
# 4) CLEAR MEMORY BEFORE PPO
# =========================================================
del rm
del rm_trainer
del tok_rm
clear_memory()


In [ ]:
# =========================================================
# 5) PPO TOKENIZER
# =========================================================
sft_name = "Qwen/Qwen2.5-0.5B-Instruct"

tok = AutoTokenizer.from_pretrained(sft_name)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Important for decoder-only models
tok.padding_side = "left"

In [ ]:
# =========================================================
# 6) LOAD PPO MODELS IN LOW MEMORY MODE
# =========================================================
# Policy model
policy = AutoModelForCausalLM.from_pretrained(
    sft_name,
    torch_dtype=dtype,
)
policy.config.pad_token_id = tok.pad_token_id
policy.gradient_checkpointing_enable()

# Some setups need this with checkpointing
if hasattr(policy, "enable_input_require_grads"):
    policy.enable_input_require_grads()

# Reference model
ref_model = AutoModelForCausalLM.from_pretrained(
    sft_name,
    torch_dtype=dtype,
)
ref_model.config.pad_token_id = tok.pad_token_id

# Reward model
reward_model = AutoModelForSequenceClassification.from_pretrained(
    "./rm_out",
    num_labels=1,
    torch_dtype=dtype,
)
reward_model.config.pad_token_id = tok.pad_token_id

# Value model
value_model = AutoModelForSequenceClassification.from_pretrained(
    "./rm_out",
    num_labels=1,
    torch_dtype=dtype,
)
value_model.config.pad_token_id = tok.pad_token_id

clear_memory()

In [ ]:

# =========================================================
# 7) PPO DATASET
# =========================================================
prompt_ds = Dataset.from_list([
    {"prompt": "How should I prepare for exams?"},
    {"prompt": "How do I improve fitness?"},
    {"prompt": "What is AI?"},
])

def tokenize_for_ppo(example):
    encoded = tok(
        example["prompt"],
        padding=False,
        truncation=True,
        max_length=128,
    )
    input_ids = encoded["input_ids"]
    return {
        "input_ids": input_ids,
        "lengths": len(input_ids),
    }

ppo_train_dataset = prompt_ds.map(
    tokenize_for_ppo,
    remove_columns=prompt_ds.column_names,
)

ppo_train_dataset = ppo_train_dataset.filter(lambda x: x["lengths"] <= 128)

In [ ]:
# =========================================================
# 8) PPO CONFIG (VERY LIGHT)
# =========================================================
ppo_args = PPOConfig(
    output_dir="./ppo_out",
    per_device_train_batch_size=1,
    learning_rate=1e-6,
    num_mini_batches=1,
    total_episodes=4,     # very small for demo
    response_length=8,    # very small for memory saving
    report_to="none",
)


In [ ]:
# =========================================================
# 9) PPO TRAINER
# =========================================================
ppo_trainer = PPOTrainer(
    args=ppo_args,
    processing_class=tok,
    model=policy,
    ref_model=ref_model,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ppo_train_dataset,
)

print("PPOTrainer initialized successfully!")

In [ ]:
# =========================================================
# 10) TRAIN PPO
# =========================================================
print("=== Training Policy with PPO ===")
ppo_trainer.train()

In [ ]:
# =========================================================
# 11) SAVE FINAL MODEL
# =========================================================
ppo_trainer.save_model("./ppo_out")
tok.save_pretrained("./ppo_out")

print("Training completed and model saved to ./ppo_out")